# راهنمای جامع طراحی سیستم‌های گفتگو
## Dialogue Systems — A Practical Tutorial

این نوت‌بوک یک راهنمای کامل و کاربردی برای ساخت سیستم گفتگو است.

| بخش | موضوع |
|-----|-------|
| ۱ | نصب کتابخانه‌ها و لایه ترجمه (پایه همه چیز) |
| ۲ | معماری GUS و مفهوم فریم |
| ۳ | پیاده‌سازی `DialogueFrame` |
| ۴ | استخراج قصد و موجودیت (NLP) |
| ۵ | مدیریت گفتگو (`DialogueManager`) |
| ۶ | سرویس‌های واقعی + API آب‌وهوا |
| ۷ | تشخیص هوشمند قصد (Semantic) |
| ۸ | ایجنت چندزبانه (Multilingual) |
| ۹ | چت تعاملی نهایی |


## بخش ۱: نصب و راه‌اندازی

### کتابخانه‌های مورد نیاز
| کتابخانه | کاربرد |
|----------|--------|
| `re` | جستجوی الگوی متنی (پیش‌فرض Python) |
| `requests` | اتصال به API آب‌وهوا |
| `deep-translator` | ترجمه رایگان به/از هر زبان |
| `langdetect` | تشخیص خودکار زبان |

> **مهم:** سلول زیر را اول اجرا کنید تا همه کتابخانه‌ها نصب شوند.


In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

libs = {'deep-translator': 'deep_translator', 'langdetect': 'langdetect', 'requests': 'requests'}
for pip_name, import_name in libs.items():
    try:
        __import__(import_name)
        print(f'  OK  {pip_name}')
    except ImportError:
        print(f'  Installing {pip_name}...')
        install(pip_name)
        print(f'  Done {pip_name}')

import re, random, requests
from deep_translator import GoogleTranslator
from langdetect import detect as detect_lang
print('\nAll libraries loaded.')


  OK  deep-translator
  OK  langdetect
  OK  requests

All libraries loaded.


### ۱.۱ لایه ترجمه — پایه همه چیز

این مهم‌ترین اصل طراحی این سیستم است:

```
ورودی کاربر (هر زبانی)
    ↓  تشخیص زبان
    ↓  ترجمه به انگلیسی
    ↓  پردازش داخلی (همه به انگلیسی)
    ↓  ترجمه جواب به زبان کاربر
    ↓  نمایش
```

با این معماری، تمام کلیدواژه‌ها، الگوها و منطق داخلی فقط به **انگلیسی** نوشته می‌شوند.
کاربر می‌تواند به فارسی، سوئدی، آلمانی یا هر زبان دیگری بنویسد.


In [2]:
class TranslationLayer:
    """Detects language, translates input to EN, translates output back."""

    SUPPORTED = {
        'fa': 'Persian', 'en': 'English', 'sv': 'Swedish',
        'de': 'German', 'fr': 'French', 'ar': 'Arabic', 'es': 'Spanish', 'uk': 'Ukrainian'
    }

    def __init__(self):
        self.user_lang = 'en'

    def to_english(self, text):
        try:
            raw = detect_lang(text)
            # Urdu (ur) and Persian (fa) share the Arabic script — prefer fa
            self.user_lang = 'fa' if raw == 'ur' else raw
        except Exception:
            self.user_lang = 'en'
        if self.user_lang == 'en':
            return text
        try:
            translated = GoogleTranslator(source=self.user_lang, target='en').translate(text)
            lang_name = self.SUPPORTED.get(self.user_lang, self.user_lang)
            print(f'  [Translate] {lang_name} → EN: "{translated}"')
            return translated
        except Exception:
            return text  # fallback: original text

    def from_english(self, text):
        if self.user_lang == 'en':
            return text
        try:
            return GoogleTranslator(source='en', target=self.user_lang).translate(text)
        except Exception:
            return text

# Quick test
tl = TranslationLayer()
fa_sample = 'فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟'
en_sample = tl.to_english(fa_sample)
print(f'  Original : {fa_sample}')
print(f'  English  : {en_sample}')
print(f'  Back (FA): {tl.from_english("Weather in stockholm: 14C, cloudy. Wear an extra layer.")}')


  [Translate] Persian → EN: "I want to go to Stockholm tomorrow, what should I wear?"
  Original : فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟
  English  : I want to go to Stockholm tomorrow, what should I wear?
  Back (FA): آب و هوا در استکهلم: 14 درجه سانتیگراد، ابری. یک لایه اضافی بپوشید.


---
## بخش ۲: معماری GUS و مفهوم فریم

سیستم **GUS** (Genial Understander System, 1977) اولین سیستم گفتگوی هدف‌محور بود.

ایده اصلی: هر **وظیفه** (مثل رزرو پرواز) را به صورت یک **فریم** نمایش بده:

```
Frame: WeatherQuery
  Slot: location  → [empty]
  Slot: date      → [empty]
```

سیستم یک به یک Slotها را از کاربر می‌پرسد تا همه پر شوند (**Slot Filling**).
این معماری را **Frame-based** یا **Task-oriented Dialogue** می‌نامند.


---
## بخش ۳: پیاده‌سازی کلاس فریم

هر سرویس (آب‌وهوا، رستوران، حمل‌ونقل) یک `DialogueFrame` است.

> **کتابخانه:** Python خالص — شی‌گرایی استاندارد


In [3]:
class DialogueFrame:
    """Represents one service task with named slots to fill."""

    def __init__(self, name, slots):
        self.name = name
        self.slots = {slot: None for slot in slots}

    def is_complete(self):
        return all(v is not None for v in self.slots.values())

    def get_missing_slot(self):
        return next((s for s, v in self.slots.items() if v is None), None)

    def reset(self):
        self.slots = {s: None for s in self.slots}

    def __repr__(self):
        return f'Frame({self.name}, {self.slots})'


def get_initial_frames():
    return {
        'weather':    DialogueFrame('Weather',    ['location', 'date']),
        'restaurant': DialogueFrame('Restaurant', ['cuisine', 'location', 'price_range']),
        'transport':  DialogueFrame('Transport',  ['origin', 'destination', 'time']),
    }

frames = get_initial_frames()
print('Frames created:')
for name, f in frames.items():
    print(f'  {name}: slots = {list(f.slots.keys())}')


Frames created:
  weather: slots = ['location', 'date']
  restaurant: slots = ['cuisine', 'location', 'price_range']
  transport: slots = ['origin', 'destination', 'time']


---
## بخش ۴: استخراج قصد و موجودیت (NLP)

چون **لایه ترجمه** همه ورودی‌ها را به انگلیسی تبدیل می‌کند،
تمام کلیدواژه‌ها فقط **انگلیسی** هستند — ساده‌تر و قوی‌تر.

- **Intent**: هدف کاربر → کدام سرویس؟
- **Entity**: مقادیر خاص → کجا؟ کِی؟ چه غذایی؟

> **کتابخانه:** `re` — جستجوی الگو با `re.IGNORECASE` و پترن‌های ترکیبی


In [4]:
# ── English-only keywords (translation layer handles other languages) ──────
KEYWORDS = {
    'intents': {
        'weather':    ['weather', 'forecast', 'temperature', 'climate'],
        'restaurant': ['restaurant', 'food', 'eat', 'hungry', 'dinner', 'lunch', 'meal', 'cafe'],
        'transport':  ['bus', 'train', 'flight', 'ticket', 'travel', 'commute', 'trip'],
    },
    'entities': {
        'location':    ['london', 'tehran', 'paris', 'stockholm', 'berlin', 'amsterdam',
                        'rome', 'madrid', 'dubai', 'tokyo', 'new york', 'sydney',
                        'gothenburg', 'vienna', 'zurich', 'oslo', 'copenhagen'],
        'date':        ['today', 'tomorrow', 'monday', 'tuesday', 'wednesday',
                        'thursday', 'friday', 'saturday', 'sunday', 'weekend', 'next week'],
        'cuisine':     ['italian', 'pizza', 'persian', 'kebab', 'chinese', 'japanese',
                        'indian', 'mexican', 'french', 'sushi', 'burger', 'thai'],
        'price_range': ['cheap', 'budget', 'affordable', 'expensive', 'luxury', 'fine dining'],
        'origin':      ['home', 'airport', 'hotel', 'station', 'office'],
        'destination': ['work', 'office', 'city center', 'downtown', 'airport', 'hotel', 'station'],
        'time':        ['morning', 'afternoon', 'evening', 'night', '8am', '10am', '12pm', '8pm'],
    }
}

def extract_info(text):
    """Extract intent and entities from English text using regex."""
    found_intent = None
    found_entities = {}

    for intent, words in KEYWORDS['intents'].items():
        if re.search('|'.join(re.escape(w) for w in words), text, re.IGNORECASE):
            found_intent = intent
            break

    for etype, examples in KEYWORDS['entities'].items():
        m = re.search('|'.join(re.escape(e) for e in examples), text, re.IGNORECASE)
        if m:
            found_entities[etype] = m.group(0).lower()

    return found_intent, found_entities


In [5]:
# ── Test extract_info ────────────────────────────────────────────────────────
tests = [
    'I want cheap pizza in stockholm tomorrow',
    'What is the weather forecast for berlin this weekend?',
    'Book a flight from airport to city center in the morning',
]
print('NLP Test Results:')
print('-' * 55)
for t in tests:
    intent, entities = extract_info(t)
    print(f'  Input:    {t}')
    print(f'  Intent:   {intent}')
    print(f'  Entities: {entities}')
    print()


NLP Test Results:
-------------------------------------------------------
  Input:    I want cheap pizza in stockholm tomorrow
  Intent:   None
  Entities: {'location': 'stockholm', 'date': 'tomorrow', 'cuisine': 'pizza', 'price_range': 'cheap'}

  Input:    What is the weather forecast for berlin this weekend?
  Intent:   weather
  Entities: {'location': 'berlin', 'date': 'weekend'}

  Input:    Book a flight from airport to city center in the morning
  Intent:   transport
  Entities: {'origin': 'airport', 'destination': 'airport', 'time': 'morning'}



---
## بخش ۵: مدیریت گفتگو (`DialogueManager`)

این کلاس «مغز» سیستم است. در هر پیام تصمیم می‌گیرد:
1. کدام سرویس فعال شود؟
2. چه اطلاعاتی جمع شده؟
3. چه چیزی هنوز کم است؟ → سوال بپرس
4. همه چیز کامل شد؟ → سرویس را صدا بزن

> **اصلاح باگ:** نسخه قبلی هنگام Intent ناشناخته `KeyError` می‌داد.
> این نسخه با `if intent in self.frames` آن را برطرف می‌کند.


In [6]:
# Natural-language questions for each slot (translate much better!)
SLOT_QUESTIONS = {
    'location':    'Which city are you asking about?',
    'date':        'What date? (e.g. today, tomorrow, friday)',
    'cuisine':     'What type of food? (e.g. pizza, sushi, kebab)',
    'price_range': 'What is your budget? (cheap / moderate / luxury)',
    'origin':      'Where are you starting from? (e.g. home, airport)',
    'destination': 'Where do you want to go? (e.g. city center, office)',
    'time':        'What time? (e.g. morning, 8am, evening)',
}

class DialogueManager:
    """Base dialogue manager: collects slots, then calls a service handler."""

    def __init__(self, frames, handlers=None):
        self.frames   = frames
        self.handlers = handlers or {}
        self.active_frame = None

    def process_input(self, english_text):
        """Process English text and return an English response."""
        intent, entities = extract_info(english_text)

        if intent and intent in self.frames:
            if not self.active_frame or intent != self.active_frame.name.lower():
                self.active_frame = self.frames[intent]
                self.active_frame.reset()
                print(f'  [System] Service activated: {self.active_frame.name}')

        if not self.active_frame:
            return 'Hello! I can help with weather, restaurants, or transport. What do you need?'

        for etype, val in entities.items():
            if etype in self.active_frame.slots:
                self.active_frame.slots[etype] = val

        if self.active_frame.is_complete():
            fname     = self.active_frame.name.lower()
            slots_now = dict(self.active_frame.slots)
            self.active_frame.reset()
            self.active_frame = None
            if fname in self.handlers:
                return self.handlers[fname](slots_now)
            return f'Request registered: {slots_now}'
        else:
            missing  = self.active_frame.get_missing_slot()
            question = SLOT_QUESTIONS.get(missing, f"Please tell me the '{missing}'.")
            return question


---
## بخش ۶: سرویس‌های واقعی

### ۶.۱ API آب‌وهوا (OpenWeatherMap)
برای دریافت اطلاعات واقعی آب‌وهوا از [openweathermap.org](https://openweathermap.org):
1. ثبت‌نام رایگان کنید
2. به بخش **API Keys** بروید و کلید خود را کپی کنید
3. آن را در متغیر `OWM_API_KEY` قرار دهید

> اگر کلید ندارید، سیستم به صورت خودکار از داده شبیه‌سازی‌شده استفاده می‌کند.

### ۶.۲ رستوران و حمل‌ونقل
در یک پروژه واقعی از Google Places API و Google Maps API استفاده می‌شود.
در اینجا از داده شبیه‌سازی‌شده استفاده می‌کنیم.


In [7]:
# ── OpenWeatherMap API key ───────────────────────────────────────────────────
# اگر کلید ندارید، خالی بگذارید — سیستم از Mock استفاده می‌کند
OWM_API_KEY = ''  # ← کلید خود را اینجا بگذارید

def service_weather(slots):
    location = slots.get('location', 'london')
    date     = slots.get('date', 'today')

    # ── Real API call ────────────────────────────────────────────────────────
    if OWM_API_KEY:
        try:
            url = (f'https://api.openweathermap.org/data/2.5/weather'
                   f'?q={location}&appid={OWM_API_KEY}&units=metric')
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                d    = r.json()
                temp = d['main']['temp']
                desc = d['weather'][0]['description']
                feel = d['main']['feels_like']
                if temp < 5:
                    advice = 'Wear a heavy coat and gloves.'
                elif temp < 12:
                    advice = 'Wear a warm jacket.'
                elif 'rain' in desc:
                    advice = 'Take an umbrella.'
                else:
                    advice = 'Dress comfortably.'
                return (f'[Live] Weather in {location.title()} ({date}): '
                        f'{temp:.1f}°C (feels like {feel:.1f}°C) | '
                        f'{desc.capitalize()} | {advice}')
            else:
                print(f'  [OWM] API error {r.status_code}, using mock data.')
        except Exception as e:
            print(f'  [OWM] Connection error: {e}, using mock data.')

    # ── Mock fallback ────────────────────────────────────────────────────────
    conditions = [
        ('sunny',  22, 'Dress comfortably.'),
        ('cloudy', 14, 'Wear an extra layer.'),
        ('rainy',  10, 'Take an umbrella.'),
        ('snowy',  -2, 'Wear a heavy coat and gloves.'),
    ]
    cond, temp, advice = random.choice(conditions)
    return (f'[Mock] Weather in {location.title()} ({date}): '
            f'{temp}°C | {cond.capitalize()} | {advice}')


def service_restaurant(slots):
    cuisine  = slots.get('cuisine', 'local')
    location = slots.get('location', 'city center')
    price    = slots.get('price_range', 'moderate')
    options  = {
        'pizza':    ['Pizza Palace', 'Napoli Star', 'Roma Express'],
        'kebab':    ['Kebab King', 'Persian Grill', 'Shiraz Garden'],
        'italian':  ['La Bella Italia', 'Venezia Ristorante', 'Trattoria Roma'],
        'sushi':    ['Tokyo Garden', 'Sakura', 'Zen Kitchen'],
        'burger':   ['Burger Lab', 'The Grill House', 'Stack & Smash'],
    }
    chosen = random.choice(options.get(cuisine, ['The Local Bistro', 'City Eats']))
    stars  = random.randint(4, 5)
    return (f'[Mock] Best {cuisine} restaurant in {location.title()} ({price}): '
            f'{chosen} | Rating: {stars}/5 ⭐')


def service_transport(slots):
    origin   = slots.get('origin', 'your location')
    dest     = slots.get('destination', 'destination')
    time_val = slots.get('time', 'any time')
    price    = random.randint(2, 20)
    duration = random.randint(10, 60)
    return (f'[Mock] Route: {origin} → {dest} | '
            f'Departure: {time_val} | Duration: {duration} min | '
            f'Price: €{price}')


SERVICE_HANDLERS = {
    'weather':    service_weather,
    'restaurant': service_restaurant,
    'transport':  service_transport,
}

# Quick weather test
dm_test = DialogueManager(get_initial_frames(), SERVICE_HANDLERS)
print('Service handlers ready.')
if OWM_API_KEY:
    print('  OWM API key found — will use live weather data.')
else:
    print('  No OWM API key — will use mock weather data.')


Service handlers ready.
  No OWM API key — will use mock weather data.


### ۶.۳ دمو چند مرحله‌ای
در زیر می‌بینید که چگونه سیستم قدم به قدم Slotها را پر می‌کند و در نهایت سرویس را صدا می‌زند.


In [8]:
print('=' * 60)
print('Demo: Multi-step Slot Filling')
print('=' * 60)

scenarios = [
    {'title': 'Weather in Stockholm tomorrow',
     'msgs':  ['weather', 'stockholm', 'tomorrow']},
    {'title': 'Cheap pizza restaurant in London',
     'msgs':  ['restaurant', 'pizza', 'london', 'cheap']},
    {'title': 'Morning bus from airport',
     'msgs':  ['bus ticket', 'airport', 'city center', 'morning']},
]

for sc in scenarios:
    dm = DialogueManager(get_initial_frames(), SERVICE_HANDLERS)
    print(f"\nScenario: {sc['title']}")
    print('-' * 50)
    for msg in sc['msgs']:
        print(f'  User:  {msg}')
        resp = dm.process_input(msg)
        print(f'  Agent: {resp}')


Demo: Multi-step Slot Filling

Scenario: Weather in Stockholm tomorrow
--------------------------------------------------
  User:  weather
  [System] Service activated: Weather
  Agent: Which city are you asking about?
  User:  stockholm
  Agent: What date? (e.g. today, tomorrow, friday)
  User:  tomorrow
  Agent: [Mock] Weather in Stockholm (tomorrow): -2°C | Snowy | Wear a heavy coat and gloves.

Scenario: Cheap pizza restaurant in London
--------------------------------------------------
  User:  restaurant
  [System] Service activated: Restaurant
  Agent: What type of food? (e.g. pizza, sushi, kebab)
  User:  pizza
  Agent: Which city are you asking about?
  User:  london
  Agent: What is your budget? (cheap / moderate / luxury)
  User:  cheap
  Agent: [Mock] Best pizza restaurant in London (cheap): Pizza Palace | Rating: 4/5 ⭐

Scenario: Morning bus from airport
--------------------------------------------------
  User:  bus ticket
  [System] Service activated: Transport
  Agent: 

---
## بخش ۷: تشخیص هوشمند قصد (Semantic Intent Detection)

### مشکل
> *«I want to go to Stockholm tomorrow, what should I wear?»*

هیچ کلمه‌ای از `weather` یا `forecast` در این جمله نیست —
سیستم قبلی چیزی نمی‌فهمید.

### راه‌حل: نگاشت مفهومی (Concept Mapping)
برای هر Intent، مفاهیم مرتبط (Synonyms & Related Concepts) تعریف می‌کنیم:

| مفهوم ورودی | Intent شناسایی‌شده |
|-------------|--------------------|
| wear, outfit, cold, coat | weather |
| hungry, starving, dinner | restaurant |
| going to, heading to + location | transport/weather |


In [9]:
# Concepts strongly associated with weather (override transport if found)
WEAR_CONCEPTS = [
    'wear', 'wearing', 'outfit', 'dress', 'clothes', 'clothing',
    'coat', 'jacket', 'umbrella', 'hot outside', 'cold outside',
]

CONCEPT_MAP = {
    'weather': [
        'temperature', 'degrees', 'snow', 'rain', 'sunny', 'cloudy',
        'freezing', 'chilly', 'windy', 'humid',
    ],
    'restaurant': [
        'hungry', 'starving', 'craving', 'brunch', 'takeaway',
        'takeout', 'delivery', 'snack',
    ],
    'transport': [
        'commute', 'arrive', 'departure', 'schedule', 'route',
        'get to', 'heading to',
    ],
}

EXTENDED_LOCATIONS = [
    'london', 'tehran', 'paris', 'stockholm', 'berlin', 'amsterdam',
    'rome', 'madrid', 'dubai', 'tokyo', 'new york', 'sydney',
    'gothenburg', 'vienna', 'zurich', 'oslo', 'copenhagen', 'brussels',
    'lisbon', 'athens', 'budapest', 'warsaw', 'prague', 'helsinki',
]

def semantic_extract_info(text):
    """Enhanced extract: keywords, then clothing override, then concept map, then extended locations."""
    intent, entities = extract_info(text)  # Step 1: standard keywords

    # Step 2: clothing context → always weather (highest priority override)
    wear_pattern = '|'.join(re.escape(w) for w in WEAR_CONCEPTS)
    if re.search(wear_pattern, text, re.IGNORECASE):
        if intent != 'weather':
            print(f'  [Semantic] Clothing context detected → overriding to weather.')
        intent = 'weather'

    # Step 3: concept mapping (only if still no intent)
    if not intent:
        for i, concepts in CONCEPT_MAP.items():
            if re.search('|'.join(re.escape(c) for c in concepts), text, re.IGNORECASE):
                intent = i
                print(f'  [Semantic] Intent "{intent}" inferred from context.')
                break

    # Step 4: extended location list
    if 'location' not in entities:
        m = re.search('|'.join(re.escape(l) for l in EXTENDED_LOCATIONS), text, re.IGNORECASE)
        if m:
            entities['location'] = m.group(0).lower()
            print(f'  [Semantic] Location "{entities["location"]}" detected.')

    # Step 5: extended date detection
    if 'date' not in entities:
        date_words = ['today', 'tomorrow', 'weekend', 'next week',
                      'monday','tuesday','wednesday','thursday','friday','saturday','sunday']
        m = re.search('|'.join(re.escape(d) for d in date_words), text, re.IGNORECASE)
        if m:
            entities['date'] = m.group(0).lower()

    return intent, entities


class SemanticDialogueManager(DialogueManager):
    """Uses semantic_extract_info for richer understanding."""

    def process_input(self, english_text):
        intent, entities = semantic_extract_info(english_text)
        if intent and intent in self.frames:
            if not self.active_frame or intent != self.active_frame.name.lower():
                self.active_frame = self.frames[intent]
                self.active_frame.reset()
                print(f'  [System] Service activated: {self.active_frame.name}')
        if not self.active_frame:
            return 'Hello! I can help with weather, restaurants, or transport.'
        for etype, val in entities.items():
            if etype in self.active_frame.slots:
                self.active_frame.slots[etype] = val
        if self.active_frame.is_complete():
            fname, slots_now = self.active_frame.name.lower(), dict(self.active_frame.slots)
            self.active_frame.reset(); self.active_frame = None
            return self.handlers.get(fname, lambda s: str(s))(slots_now)
        missing  = self.active_frame.get_missing_slot()
        return SLOT_QUESTIONS.get(missing, f"Please tell me the '{missing}'.")


print('SemanticDialogueManager ready.')


SemanticDialogueManager ready.


In [10]:
print('=' * 60)
print('Demo: Semantic Intent Detection')
print('=' * 60)

implicit_tests = [
    'I want to go to Stockholm tomorrow, what should I wear?',
    'heading to Berlin this weekend — hot or cold?',
    'I am starving, where can I eat sushi in london?',
    'need to get from airport to city center in the morning',
]

for query in implicit_tests:
    dm = SemanticDialogueManager(get_initial_frames(), SERVICE_HANDLERS)
    print(f'\nUser:  {query}')
    print(f'Agent: {dm.process_input(query)}')
    print('-' * 50)


Demo: Semantic Intent Detection

User:  I want to go to Stockholm tomorrow, what should I wear?
  [Semantic] Clothing context detected → overriding to weather.
  [System] Service activated: Weather
Agent: [Mock] Weather in Stockholm (tomorrow): 14°C | Cloudy | Wear an extra layer.
--------------------------------------------------

User:  heading to Berlin this weekend — hot or cold?
  [Semantic] Intent "transport" inferred from context.
  [System] Service activated: Transport
Agent: Where are you starting from? (e.g. home, airport)
--------------------------------------------------

User:  I am starving, where can I eat sushi in london?
  [System] Service activated: Restaurant
Agent: What is your budget? (cheap / moderate / luxury)
--------------------------------------------------

User:  need to get from airport to city center in the morning
Agent: Hello! I can help with weather, restaurants, or transport.
--------------------------------------------------


---
## بخش ۸: ایجنت چندزبانه (Multilingual Agent)

حالا همه اجزا را ترکیب می‌کنیم:

```
      کاربر (هر زبانی)
           ↓
    TranslationLayer.to_english()
           ↓
  SemanticDialogueManager.process_input()
      (english keywords only)
           ↓
    service_weather / restaurant / transport
      (real OWM API or mock)
           ↓
    TranslationLayer.from_english()
           ↓
      پاسخ به زبان کاربر
```


In [11]:
class MultilingualAgent:
    """Full pipeline: any language → English processing → response in user language."""

    def __init__(self):
        self.translator = TranslationLayer()
        self.dm = SemanticDialogueManager(get_initial_frames(), SERVICE_HANDLERS)

    def chat(self, user_text):
        en_input   = self.translator.to_english(user_text)
        en_response = self.dm.process_input(en_input)
        return self.translator.from_english(en_response)

    def reset(self):
        self.dm = SemanticDialogueManager(get_initial_frames(), SERVICE_HANDLERS)
        self.translator = TranslationLayer()


print('=' * 60)
print('Demo: MultilingualAgent — same question in 4 languages')
print('=' * 60)

same_question = [
    ('Persian',  'فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟'),
    ('Swedish',  'Vad ska jag ha pa mig i Stockholm imorgon?'),
    ('German',   'Was soll ich morgen in Stockholm anziehen?'),
    ('English',  'What should I wear in Stockholm tomorrow?'),
]

for lang, question in same_question:
    agent = MultilingualAgent()
    print(f'\n[{lang}] User:  {question}')
    print(f'[{lang}] Agent: {agent.chat(question)}')
    print('-' * 55)


Demo: MultilingualAgent — same question in 4 languages

[Persian] User:  فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟
  [Translate] Persian → EN: "I want to go to Stockholm tomorrow, what should I wear?"
  [Semantic] Clothing context detected → overriding to weather.
  [System] Service activated: Weather
[Persian] Agent: [مسخره] آب و هوا در استکهلم (فردا): 10 درجه سانتی گراد | بارانی | یک چتر بردارید
-------------------------------------------------------

[Swedish] User:  Vad ska jag ha pa mig i Stockholm imorgon?
  [Translate] Swedish → EN: "What should I wear in Stockholm tomorrow?"
  [Semantic] Clothing context detected → overriding to weather.
  [System] Service activated: Weather
[Swedish] Agent: [Mock] Vädret i Stockholm (i morgon): 22°C | Soligt | Klä dig bekvämt.
-------------------------------------------------------

[German] User:  Was soll ich morgen in Stockholm anziehen?
  [Translate] German → EN: "What should I wear in Stockholm tomorrow?"
  [Semantic] Clothing contex

---
## بخش ۹: چت تعاملی نهایی

حالا می‌توانید واقعاً با ایجنت صحبت کنید!

- به **هر زبانی** بنویسید
- جواب را در **همان زبان** دریافت کنید
- برای خروج بنویسید: `exit`

> خط `start_interactive_chat()` را از حالت کامنت خارج کنید.


In [ ]:
def start_interactive_chat():
    """Interactive multilingual chat session."""
    agent = MultilingualAgent()
    print('Agent: Hello! / سلام! / Hola! / Hallo! / Bonjour!')
    print("      (Type in any language. Type 'exit' to quit)")
    print('-' * 50)
    while True:
        user_input = input('You: ').strip()
        if not user_input:
            continue
        if user_input.lower() in ('exit', 'quit', 'خروج', 'avsluta'):
            print('Agent: Goodbye! / خداحافظ! / Auf Wiedersehen!')
            break
        response = agent.chat(user_input)
        print(f'Agent: {response}\n')

# خط زیر را از حالت کامنت خارج کنید تا چت شروع شود:
start_interactive_chat()


Agent: Hello! / سلام! / Hola! / Hallo! / Bonjour!
      (Type in any language. Type 'exit' to quit)
--------------------------------------------------
  [Translate] Persian → EN: "I want to travel to Gothenburg tomorrow, what should I wear?"
  [Semantic] Clothing context detected → overriding to weather.
  [System] Service activated: Weather
Agent: [مسخ] آب و هوا در گوتنبرگ (فردا): 10 درجه سانتی گراد | بارانی | یک چتر بردارید

  [Translate] Persian → EN: "hello"
Agent: سلام! من می توانم در مورد آب و هوا، رستوران ها یا حمل و نقل کمک کنم.

